<a href="https://colab.research.google.com/github/deji4things2000/mlpro/blob/master/ps1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# Initialize Otter
import otter
grader = otter.Notebook("ps1.ipynb")

ModuleNotFoundError: No module named 'otter'

# PSET 1: Optimizing Matrix Multiplication & CUDA

...

Try not to add additional cells nor rearrange any that exist now as it will mess with our autograder (but feel free to open another Colab/notebook on the side).

### Make sure to submit your final notebook with all of your solutions to Gradescope!
[Direct Gradescope Link](https://www.gradescope.com/courses/1048026)

### Before Starting

This problem set uses NVCC (NVIDIA CUDA Compiler) within Google Colab to compile and run CUDA code. Google Colab provides a convenient (and free!) environment with GPU support for executing these programs.  At the top right corner, under the arrow dropdown next to the `Connect` button, select "Change runtime type" and choose `T4 GPU` for this problem set.  Note, across your account, you can only have one runtime connected to a GPU at a given time.

We're in an interactive _Python_ notebook.  So, of course, running C++ code is not going to be as easy as it is to run Python code.  That said, we'll once again use helpers to make this cleaner.

Our plugins `%%cpurun` and `%%gpurun` save, compile, and run your C++ and/or CUDA code in the cell.  This is different than before where the helper simply *saved* the file, and then we would compile using `g++` and run it ourselves.  But now, since you remember the nuances of working with a compiled language, we'll just bundle it all up.

In [1]:
# make sure CUDA is installed
!nvcc --version

# make sure you have a GPU runtime (if this fails go to runtime -> change runtime type)
!nvidia-smi

# Install some magic to run and save .cpp programs
!curl -o ./cpu_runner.py https://raw.githubusercontent.com/COSC-169-23-F25/helpers/main/cpu_runner.py
%load_ext cpu_runner

# Install some magic to run and save .cu C++ CUDA programs
!curl -o ./gpu_runner.py https://raw.githubusercontent.com/COSC-169-23-F25/helpers/main/gpu_runner.py
%load_ext gpu_runner

# to learn about how to do more fancy things with CUDA using this API see:
# https://nvcc4jupyter.readthedocs.io/en/latest/index.html

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Fri Sep 26 22:36:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8       

One other note: we import a few `.o` "dot-oh" object and `.h` header files to bring along some printing utilities and fuzz-testing in an obscured way (we wouldn't want to give away the solutions 😛).

As you step away and return to this problem set, you will often find that Colab has disconnected your runtime and deleted any files.  You'll need to re-run cells like the above which import the magic compilation utilities as well as the cells with `curl` imports of the various `.o` and `.h` files.

## Problem 1: Python Matrix Multiply (again)
In the last problem set, you explored how to write a simple matrix multiplication in Python using a naïve implementation for representing matrices using lists of lists.

In `C++` there are ways to mimic this style:
- multi-dimensional arrays like `int my_matrix[N][M]`
- `std::vector<std::vector<float> >`

But it gets quite a bit more difficult when we try to do this using CUDA.  An easier way to handle such a problem is to put everything into one contiguous chunk of memory in an organized way.  As we've discussed in class, there are two standard methods for representing/organizing our matrix in memory:
1. row-major order
2. column-major order

These are visualized below:
![row-major](https://raw.githubusercontent.com/COMS-BC3159-F24/image_assets/main/row-col-major.png)

In the written part of PS0, you reacquainted yourself with the standard matrix multiplication algorithm:
$$C_{ij} = \sum_{k=1}^{n} A_{ik} \times B_{kj}$$

Then, in the coding part of PS0, using `list`s of `list`s, you computed the product of a matrix $A$ (in gray) which is $4\times5$ and $B$ (blue) of dimensions $5\times4$ whose product results in $C$ (purple) of size $4\times4$.

![matrix mult example](https://raw.githubusercontent.com/COMS-BC3159-F24/image_assets/main/matmul_example.png)

This time, write a matrix multiplication implementation that indexes into provided **row-major order** matrices to compute their product.  As before, we've provided the matrices (note: they're _not_ `list`s of `list`s this time; instead, just one flat `list`) and some printing functions (albeit a bit obscured to leave the challenge of figuring out the indexing for the actual multiplication up to you)!

In [2]:
# Helper function to print a matrix represented as a flat list in a row-major order
def printFlatMatrix(matrix, num_rows, num_cols, width=6):
    for i in range(num_rows):
        formatted_row = ''.join(f'{matrix[j]:{width}}'
                                for j in range(
                                    i * num_cols, i * num_cols + num_cols))
        print(formatted_row)

# Matrices' dimensions (can't be inferred from list len)
rows_A, cols_A = 4, 5
rows_B, cols_B = 5, 4

# Note: single list in row-major
A = [
    0, 1, 2, 3, 4,
    5, 6, 7, 8, 9,
    10, 11, 12, 13, 14,
    15, 16, 17, 18, 19
]
B = [
    20, 21, 22, 23,
    24, 25, 26, 27,
    28, 29, 30, 31,
    32, 33, 34, 35,
    36, 37, 38, 39
]
C = [0] * (rows_A * cols_B)

In [3]:
def matrixMultiply(A, B, rows_A, cols_B):
    cols_A = len(A) // rows_A
    rows_B = len(B) // cols_B
    assert cols_A == rows_B

    C = [0] * (rows_A * cols_B)

    for i in range(rows_A):
      for j in range(cols_B):
        sum_val = 0
        for k in range(cols_A):
          sum_val+= A[i * cols_A + k] * B[k* cols_B + j]
        C[i * cols_B + j] = sum_val
    return C

C = matrixMultiply(A, B, rows_A, cols_B)
printFlatMatrix(C, rows_A, cols_B)

   320   330   340   350
  1020  1055  1090  1125
  1720  1780  1840  1900
  2420  2505  2590  2675


## C++ Malloc Practice [Ungraded]
As a warmup, let's review how we allocate memory dynamically.  Allocate memory for a row of 5 integers and fill the row with values 0-4.  Print the array, and don't forget to `free()` when you're done!

In [4]:
%%cpurun -n malloc_example.cpp
#include <cstdio>  // For printf
#include <cstdlib> // For malloc and free

int main() {
    // Allocate memory for a row of 5 integers
    int* rowA = (int*)malloc(5*sizeof(int)); // TODO: Actually allocate!
    if (rowA == nullptr) {
        std::fprintf(stderr, "Memory allocation failed\n");
        return 1;  // Exit with an error code
    }

    // Fill the array with values
    for (int i = 0; i < 5; i++) {
        rowA[i] = i;
    }

    // Alternative way to fill the array
    int* temp = rowA;
    for (int i = 0; i < 5; i++) {
        *temp = i;
        temp++; // Pointer arithmetic, increments temp pointer raw value by sizeof(int) each time!
    }

    // Print each element of the array using printf
    printf("Elements of rowA:\n");
    for (int i = 0; i < 5; i++) {
        printf("%d ", rowA[i]);
    }
    printf("\n");

    // Free allocated memory
    free(rowA);

    return 0;
}


[build]      g++ -std=c++17 malloc_example.cpp -o malloc_example
[run]        ./malloc_example
Elements of rowA:
0 1 2 3 4


## Problem 2: C++ Matrix Multiply

Before we jump to a CUDA implementation, let's continue warming up the background necessary by writing a `C++` implementation of matrix multiplication.

Although this could be done easily using `std::vector`s of `std::vector`s or multi-dimensional arrays, let's practice allocating just **one** contiguous chunk of memory per each of our matrices and use row-major ordering to organize each matrix in memory.

We've provided a `print_int_matrix()` function in the `matrix_lib.h` header.  It assumes row-major order in one contiguous chunk of memory as described above.  The function signature looks as follows:
```c
void print_int_matrix(const int* matrix, int num_rows, int num_cols)
```

We link the provided `matrix_lib.o` file via the magic `%%cpurun` compilation command. (It's obscured in this way so that we don't take the fun out of figuring out the indexing for your matrices!)

This time all the fun is left to you!  
1. Allocate the chunks of memory for $A$, $B$, and $C$.
2. Fill in $A$ and $B$ matrices as shown in the image above and the Python exercise in Problem 1.
3. Write the multiply!  (And print only $C$ via the `print_int_matrix()` function provided.)

Your output should look exactly as follows:
```text
  320   330   340   350
 1020  1055  1090  1125
 1720  1780  1840  1900
 2420  2505  2590  2675
```


In [5]:
!curl -o ./matrix_lib.o https://raw.githubusercontent.com/COMS-BC3159-F24/helpers/main/matrix_lib.o
!curl -o ./matrix_lib.h https://raw.githubusercontent.com/COMS-BC3159-F24/helpers/main/matrix_lib.h

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2336  100  2336    0     0  11466      0 --:--:-- --:--:-- --:--:-- 11507
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   234  100   234    0     0   1248      0 --:--:-- --:--:-- --:--:--  1244


In [6]:
%%cpurun -n cpp_int_matmul.cpp -o matrix_lib.o
#include <cstdio>
#include <cstdlib>
#include "matrix_lib.h"

// TODO: Write your matrix multiplication
//       where A: MxN, B: NxP, C: MxP
//       and you write C in place
void matrixMultiplyCPU_int(int* A, int* B, int* C, int M, int N, int P) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < P; j++) {
            int sum = 0;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * P + j];
            }
            C[i * P + j] = sum;
        }
    }
}



int main() {
    const int rows_A = 4, cols_A = 5;
    const int rows_B = 5, cols_B = 4;
    const int rows_C = rows_A, cols_C = cols_B;

    // Allocate matrix A
    int* A = (int*)malloc(rows_A * cols_A * sizeof(int)); // TODO!
    if (A == nullptr) {
        fprintf(stderr, "Memory allocation for A failed\n");
        return 1;
    }

    // Fill matrix A (use print function to temporarily help initialize)
    for (int i = 0; i < rows_A * cols_A; i++) {
        A[i] = i;
    }


    // Allocate matrix B
    int* B = (int*)malloc(rows_B * cols_B * sizeof(int)); // TODO!
    if (B == nullptr) {
        fprintf(stderr, "Memory allocation for B failed\n");
        free(A);
        return 1;
    }

    // Fill matrix B
    for (int i = 0; i < rows_B * cols_B; i++) {
        B[i] = 20 + i;
    }

    // Allocate matrix C (result matrix)
    int* C = (int*)malloc(rows_C * cols_C * sizeof(int)); // TODO!
    if (C == nullptr) {
        fprintf(stderr, "Memory allocation for C failed\n");
        free(A);
        free(B);
        return 1;
    }

    // Perform matrix multiplication
    matrixMultiplyCPU_int(A, B, C, rows_A, cols_A, cols_B);

    // Print result
    print_int_matrix(C, rows_A, cols_B);

    // Free allocated memory
    free(A);
    free(B);
    free(C);

    return 0;
}

[compile]      g++ -std=c++17 -c cpp_int_matmul.cpp -o cpp_int_matmul.o
[link]      g++ -o cpp_int_matmul cpp_int_matmul.o matrix_lib.o
[run]        ./cpp_int_matmul
  320   330   340   350
 1020  1055  1090  1125
 1720  1780  1840  1900
 2420  2505  2590  2675


## Problem 3: Float Matrix Multiply in C++

Before moving forward, let's run this exercise again, but using `float` instead of `int`.  In the same header, we've provided a `print_float_matrix()` for you to use (which prints 2 decimals of precision).  Your output should look exactly as follows:
```text
 320.00  330.00  340.00  350.00
1020.00 1055.00 1090.00 1125.00
1720.00 1780.00 1840.00 1900.00
2420.00 2505.00 2590.00 2675.00
```

In [7]:
%%cpurun -n cpp_float_matmul.cpp -o matrix_lib.o
#include <cstdio>
#include <cstdlib>
#include "matrix_lib.h"

// TODO: Write your matrix multiplication
//       where A: MxN, B: NxP, C: MxP
//       and you write C in place
void matrixMultiplyCPU_float(float* A, float* B, float* C, int M, int N, int P) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < P; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * P + j];
            }
            C[i * P + j] = sum;
        }
    }
}



int main() {
    const int rows_A = 4, cols_A = 5;
    const int rows_B = 5, cols_B = 4;
    const int rows_C = rows_A, cols_C = cols_B;

    // Allocate matrix A
    float* A = (float*)malloc(rows_A * cols_A * sizeof(float)); // TODO!
    if (A == nullptr) {
        fprintf(stderr, "Memory allocation for A failed\n");
        return 1;
    }

    // Fill matrix A (use print function to temporarily help initialize)
    for (int i = 0; i < rows_A * cols_A; i++) {
        A[i] = (float)i;
    }

    // Allocate matrix B
    float* B = (float*)malloc(rows_B * cols_B * sizeof(float)); // TODO!
    if (B == nullptr) {
        fprintf(stderr, "Memory allocation for B failed\n");
        free(A);
        return 1;
    }

    // Fill matrix B
   for (int i = 0; i < rows_B * cols_B; i++) {
       B[i] = 20.0f + (float)i;
   }

    // Allocate matrix C (result matrix)
    float* C = (float*)malloc(rows_C * cols_C * sizeof(float));nullptr; // TODO!
    if (C == nullptr) {
        fprintf(stderr, "Memory allocation for C failed\n");
        free(A);
        free(B);
        return 1;
    }

    // Perform matrix multiplication
    matrixMultiplyCPU_float(A, B, C, rows_A, cols_A, cols_B);

    // Print result
    print_float_matrix(C, rows_A, cols_B);

    // Free allocated memory
    free(A);
    free(B);
    free(C);

    return 0;
}

[compile]      g++ -std=c++17 -c cpp_float_matmul.cpp -o cpp_float_matmul.o
[link]      g++ -o cpp_float_matmul cpp_float_matmul.o matrix_lib.o
[run]        ./cpp_float_matmul
 320.00  330.00  340.00  350.00
1020.00 1055.00 1090.00 1125.00
1720.00 1780.00 1840.00 1900.00
2420.00 2505.00 2590.00 2675.00


In [8]:
!curl -o ./cpu_fuzz_test.o https://raw.githubusercontent.com/COMS-BC3159-F24/helpers/main/cpu_fuzz_test.o
!curl -o ./cpu_fuzz_test.h https://raw.githubusercontent.com/COMS-BC3159-F24/helpers/main/cpu_fuzz_test.h

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3992  100  3992    0     0  15084      0 --:--:-- --:--:-- --:--:-- 15121
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   324  100   324    0     0   1685      0 --:--:-- --:--:-- --:--:--  1678


In [9]:
%%cpurun -n student_testing.cpp -o matrix_lib.o,cpu_fuzz_test.o
#include "matrix_lib.h"
#include "cpu_fuzz_test.h"

// Student implementation of CUDA matrix multiplication
void matrixMultiplyCPU_float(float* A, float* B, float* C, int M, int N, int P) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < P; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * P + j];
            }
            C[i * P + j] = sum;
        }
    }
}

int main() {
    fuzzTestMatrixMultiplication(10, 50);  // Run fuzz tests
    return 0;
}


[compile]      g++ -std=c++17 -c student_testing.cpp -o student_testing.o
[link]      g++ -o student_testing student_testing.o matrix_lib.o cpu_fuzz_test.o
[run]        ./student_testing
All fuzz tests passed!


## CUDA Compiling Basics [Ungraded]
Print the following by writing a `__global__` kernel and invoking it.
```text
Hello from the GPU!
```
We'll add a `cudaDeviceSynchronize` at the end of your `main()` function to wrap up neatly!  Recall what happens if you don't include that line...(and if you don't remember, then try commenting it out)!

#### Not seeing any output?  Did you remember to change your runtime?
Don't forget to change your runtime in Google Colab to a T4 GPU.

![change runtime](https://raw.githubusercontent.com/COMS-BC3159-F24/image_assets/main/changeruntime.png)

In [10]:
%%gpurun -n my_kernel.cu
#include <cstdio>
#include <cuda_runtime.h>

// This function runs on the GPU
__global__
void helloWorldKernel() {
    printf("Hello from the GPU!!\n");
}

// This function runs on the CPU
__host__
int main() {
    // Launch the kernel on the GPU
    helloWorldKernel<<<1, 1>>>();

    // Wait for the kernel to finish executing
    cudaDeviceSynchronize();

    return 0;
}

7.5
[nvcc build]  nvcc -arch=sm_75 -o my_kernel my_kernel.cu
[run]         ./my_kernel
Hello from the GPU!!


## Problem 4: CUDA Matrix Multiply
The time has finally arrived...to "CUDA-fy" your matrix multiplication code.  You've practiced examples in class and in our HelloCUDA exercises.  Now, let's do the necessary allocations, write a kernel, move and copy data, and run!

To constrain your solution, let's use a 2D block of dimensions 2x2.  Yes, this is tiny (something more standard like 32 to collect the threads in a warp is typically sensible), but we have a small output matrix, and this is a first exercise, so why not?

You'll need to make sure your grid and respective grid dimensions support the whole matrix.  Your kernel should compute **one** element of the resulting $C$ matrix.  (We have so much compute available, so let's go parallel heavy!)

Your code from the `C++` "CPU" version of matrix multiply in Problem 3 can be used as a starting point here.  In fact, to make checking things easy, we have provided a framework of a script that leverages your `matrixMultiplyCPU_float()` from Problem 3 (just paste it here) for a reference comparison.

Follow each step provided in the "comments" `// TODO`, etc. and we've already included the print statements (using our helpers from `#include "matrix_lib.h"` again) that should provide the exact output we expect:

```text
CPU Matrix Multiply:
 320.00  330.00  340.00  350.00
1020.00 1055.00 1090.00 1125.00
1720.00 1780.00 1840.00 1900.00
2420.00 2505.00 2590.00 2675.00

GPU Matrix Multiply:
 320.00  330.00  340.00  350.00
1020.00 1055.00 1090.00 1125.00
1720.00 1780.00 1840.00 1900.00
2420.00 2505.00 2590.00 2675.00

Matrices match!
```

In [11]:
%%gpurun -n cuda_matmul.cu -o matrix_lib.o
#include <iostream>
#include <cmath>
#include <cuda_runtime.h>
#include "matrix_lib.h"

// Paste your Problem 3 solution for comparison
void matrixMultiplyCPU_float(float* A, float* B, float* C, int M, int N, int P) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < P; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * P + j];
            }
            C[i * P + j] = sum;
        }
    }
}

// CUDA kernel to multiply two matrices A[M][N] * B[N][P] = C[M][P]
// Specify what work to complete via threadIdx, blockIdx, blockDim
// Compute one element of the resulting matrix
__global__ void matrixMultiplyCUDA(float* A, float* B, float* C, int M, int N, int P) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < P) {
        float sum = 0.0f;
        for  (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * P + col];
        }
        C[row * P + col] = sum;
    }
}

// Compare matrices represented as flat arrays with single loop
// Allow for some floating-point error (GPU and CPU may differ slightly)
bool compareMatrices(const float* A, const float* B, int total_elements) {
    float epsilon = 0.001f; // Tolerance for floating-point comparison
    int precision_issues = 0;

    // Note: not traversing in row-major order
    for (int i = 0; i < total_elements; ++i) {
        float diff = fabs(A[i] - B[i]);
        if (diff > epsilon) {
            printf("Matrices differ at element %d: A[%d] = %f, B[%d] = %f\n", i, i, A[i], i, B[i]);
            return false;
        } else if (diff > epsilon / 10 && diff <= epsilon) {
            precision_issues++;
        }
    }

    if (precision_issues > 0) {
        printf("Warning: %d elements had values close to the precision threshold.\n", precision_issues);
    }

    return true;
}

int main() {
    const int rows_A = 4, cols_A = 5;
    const int rows_B = 5, cols_B = 4;
    const int rows_C = rows_A, cols_C = cols_B;

    // Allocate memory for matrices A, B, C
    // * C_gpu will be for copying and comparison purposes
    // --> You will need to make a d_ pointer to accompany the "host" h_ version
    // * C_cpu will be used to store the CPU result
    float* h_A = (float*)malloc(rows_A * cols_A * sizeof(float)); // TODO
    float* h_B = (float*)malloc(rows_B * cols_B * sizeof(float)); // TODO
    float* h_C_cpu = (float*)malloc(rows_C * cols_C * sizeof(float)); // TODO
    float* h_C_gpu = (float*)malloc(rows_C * cols_C * sizeof(float)); // TODO

    // Fill matrix A and B (similar to as you did in the CPU version)
    for (int i = 0; i < rows_A * cols_A; i++) {
        h_A[i] = (float)i;
    }

    for (int i = 0; i < rows_B * cols_B; i++) {
        h_B[i] = 20.0f + (float)i;
    }

    // Initialize C matrices to zero
    for (int i = 0; i < rows_C * cols_C; i++) {
        h_C_cpu[i] = 0.0f;
        h_C_gpu[i] = 0.0f;
    }


    // Perform matrix multiplication on CPU (C++)
    matrixMultiplyCPU_float(h_A, h_B, h_C_cpu, rows_A, cols_A, cols_B);
    printf("CPU Matrix Multiply:\n");
    print_float_matrix(h_C_cpu, rows_A, cols_B);

    // Perform matrix multiplication on GPU (CUDA)
    // create d_ device pointers, allocate GPU memory, and fill the memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, rows_A * cols_A * sizeof(float));
    cudaMalloc(&d_B, rows_B * cols_B * sizeof(float));
    cudaMalloc(&d_C, rows_C * cols_C * sizeof(float));

     // Copy data from host to device
    cudaMemcpy(d_A, h_A, rows_A * cols_A * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, rows_B * cols_B * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemset(d_C, 0, rows_C * cols_C * sizeof(float));

    // Define grid/block dimensions, then launch the kernel
    dim3 blockDim(2, 2);
    dim3 gridDim((cols_C + blockDim.x - 1) / blockDim.x,
                 (rows_C + blockDim.y - 1) / blockDim.y);

    matrixMultiplyCUDA<<<gridDim, blockDim>>>(d_A, d_B, d_C, rows_A, cols_A, cols_B);

    // Wait for kernel to complete
    cudaDeviceSynchronize();


    // Copy result back to host (h_C_gpu) for comparison
    cudaMemcpy(h_C_gpu, d_C, rows_C * cols_C * sizeof(float), cudaMemcpyDeviceToHost);

    printf("\nGPU Matrix Multiply:\n");
    print_float_matrix(h_C_gpu, rows_C, cols_B);

    // Compare the CPU and GPU results
    if (compareMatrices(h_C_cpu, h_C_gpu, rows_C*cols_C)) {
        printf("\nMatrices match!\n");
    } else {
        printf("\nMatrices do NOT match.\n");
    }

    // Free memory
    free(h_A);
    free(h_B);
    free(h_C_cpu);
    free(h_C_gpu);
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}


7.5
[nvcc build]  nvcc -arch=sm_75 -o cuda_matmul cuda_matmul.cu matrix_lib.o
[run]         ./cuda_matmul
CPU Matrix Multiply:
 320.00  330.00  340.00  350.00
1020.00 1055.00 1090.00 1125.00
1720.00 1780.00 1840.00 1900.00
2420.00 2505.00 2590.00 2675.00

GPU Matrix Multiply:
 320.00  330.00  340.00  350.00
1020.00 1055.00 1090.00 1125.00
1720.00 1780.00 1840.00 1900.00
2420.00 2505.00 2590.00 2675.00

Matrices match!


In [12]:
!curl -o ./fuzz_test.o https://raw.githubusercontent.com/COMS-BC3159-F24/helpers/main/fuzz_test.o
!curl -o ./fuzz_test.h https://raw.githubusercontent.com/COMS-BC3159-F24/helpers/main/fuzz_test.h

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  9352  100  9352    0     0  46164      0 --:--:-- --:--:-- --:--:-- 46068
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   322  100   322    0     0   1639      0 --:--:-- --:--:-- --:--:--  1642


In [14]:
%%gpurun -n student_cuda_testing.cu -o matrix_lib.o,fuzz_test.o
#include "matrix_lib.h"
#include "fuzz_test.h"

// Your CUDA matrix multiplication kernel (one element of C per thread)
__global__ void matrixMultiplyCUDA(float* A, float* B, float* C, int M, int N, int P) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < P) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * P + col];
        }
        C[row * P + col] = sum;
    }
}

int main() {
    fuzzTestMatrixMultiplication(10, 50);  // Run fuzz tests
    cudaDeviceSynchronize();
    return 0;
}


7.5
[nvcc build]  nvcc -arch=sm_75 -o student_cuda_testing student_cuda_testing.cu matrix_lib.o fuzz_test.o
[run]         ./student_cuda_testing
All fuzz tests passed!


---

## Benchmarking! [Ungraded]
Nice!  Now that you're done with the "work" for this PSET, we want to start adding some runtime numbers to this whole "optimization" concept.

In class, and more widely, we've discussed the slowness of Python *v.* C (an interpreted versus compiled language).  We've talked about the systems effects of cache and access patterns (as simple as loop orders).  And we've discussed the potential speedup offered by parallelism with the caveat that systems and memory factors may impact _more_ than parallelism can help.

Let's run some trials now to see these effects in reality!  We'll use $N \times N$ matrices where $N=2000$.

Read each cell, consider the accuracy of the timestamps, and then run (and wait...)

### Python (naïve _v._ Numpy)
This may run for a while...

In [15]:
import numpy as np
import time

def matrix_multiply(A, B):
    N = len(A)
    C = [[0] * N for _ in range(N)]
    for i in range(N):
        for j in range(N):
            for k in range(N):
                C[i][j] += A[i][k] * B[k][j]
    return C

def benchmark_matmul(N, use_numpy=False):
    # Initialize matrices with random values
    if use_numpy:
        A = np.random.randint(0, 10, (N, N))
        B = np.random.randint(0, 10, (N, N))
    else:
        A = [[np.random.randint(0, 10) for _ in range(N)] for _ in range(N)]
        B = [[np.random.randint(0, 10) for _ in range(N)] for _ in range(N)]

    # Benchmark
    start_time = time.time()

    if use_numpy:
        C = np.dot(A, B)
    else:
        C = matrix_multiply(A, B)

    end_time = time.time()

    print(f"Time taken for matrix multiplication of size {N}x{N}: {end_time - start_time:.6f} seconds (using {'NumPy' if use_numpy else 'custom function'})")

if __name__ == "__main__":
    N = 2000  # Example matrix size
    print("\nBenchmarking NumPy matrix multiplication:")
    benchmark_matmul(N, use_numpy=True)

    print("\nBenchmarking custom matrix multiplication:")
    # NOTE: only uncomment this for one test run (do not submit with this uncommented)
    # benchmark_matmul(N)



Benchmarking NumPy matrix multiplication:
Time taken for matrix multiplication of size 2000x2000: 11.269872 seconds (using NumPy)

Benchmarking custom matrix multiplication:


### C++
You'll note that the naïve C++ implementation may not necessarily beat Numpy, either!  Their optimizations (BLAS, etc.) really are excellent!

In [16]:
%%cpurun -n benchSimpleMatrixMul.cpp

#include <iostream>
#include <cstdlib>
#include <ctime>
#include <chrono>

// Define matrix size N
const int N = 2000;

// Static global arrays
int A[N][N];
int B[N][N];
int C[N][N];

void matrix_multiply() {
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            C[i][j] = 0;
            for (int k = 0; k < N; ++k) {
                C[i][j] += A[i][k] * B[k][j];
            }
        }
    }
}

void benchmark_matmul() {
    // Initialize matrices with random values
    srand(static_cast<unsigned>(time(0)));
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            A[i][j] = rand() % 10;
            B[i][j] = rand() % 10;
        }
    }

    // Benchmark
    auto start = std::chrono::high_resolution_clock::now();
    matrix_multiply();
    auto end = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double> duration = end - start;
    std::cout << "Time taken for matrix multiplication of size " << N << "x" << N << ": "
         << duration.count() << " seconds" << std::endl;
}

int main() {
    benchmark_matmul();
    return 0;
}

[build]      g++ -std=c++17 benchSimpleMatrixMul.cpp -o benchSimpleMatrixMul
[run]        ./benchSimpleMatrixMul
Time taken for matrix multiplication of size 2000x2000: 82.0132 seconds


### Compiler Optimizations

This implementation outperforms both naïve C++ and NumPy by leveraging advanced compiler optimizations such as `-Ofast`, `-funroll-loops`, and `-march=native`. These optimizations enhance performance by exploiting architecture-specific details!  Many people even [walk](https://arxiv.org/pdf/2105.07202) the large search space of optimization flags to find the best configuration.

In [ ]:
!cp benchSimpleMatrixMul.cpp benchmarkOptimizedMatrixMul.cpp
!g++ -Ofast -funroll-loops -march=native benchmarkOptimizedMatrixMul.cpp -o benchmarkOptimizedMatrixMul
!./benchmarkOptimizedMatrixMul

### Cache-Friendlier Implementations
In class, we showed that `averageMatrix()` where loop order changed could be 5x faster.  Before you dig into the differences in the code below, could you think of optimizations for the multiplication given how memory is laid out for C++?  Also consider caching and ways to share temporally local data.

![matrix mult example](https://raw.githubusercontent.com/COMS-BC3159-F24/image_assets/main/rowmajor.png)

In [17]:
%%cpurun -n cacheFriendlyMatrixMul.cpp

#include <iostream>
#include <cstdlib>
#include <ctime>
#include <chrono>

// Define matrix size N
const int N = 2000;

// Static global arrays
int A[N][N];
int B[N][N];
int C[N][N];

void matrix_multiply() {
    for (int i = 0; i < N; ++i) {
        for (int k = 0; k < N; ++k) {
            for (int j = 0; j < N; ++j) {
                C[i][j] += A[i][k] * B[k][j];
            }
        }
    }
}


void benchmark_matmul() {
    // Initialize matrices with random values
    srand(static_cast<unsigned>(time(0)));
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            A[i][j] = rand() % 10;
            B[i][j] = rand() % 10;
            C[i][j] = 0;
        }
    }

    // Benchmark
    auto start = std::chrono::high_resolution_clock::now();
    matrix_multiply();
    auto end = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double> duration = end - start;
    std::cout << "Time taken for matrix multiplication of size " << N << "x" << N << ": "
         << duration.count() << " seconds" << std::endl;
}

int main() {
    benchmark_matmul();
    return 0;
}

[build]      g++ -std=c++17 cacheFriendlyMatrixMul.cpp -o cacheFriendlyMatrixMul
[run]        ./cacheFriendlyMatrixMul
Time taken for matrix multiplication of size 2000x2000: 35.0393 seconds


Adding optimization flags, now:

In [18]:
!cp cacheFriendlyMatrixMul.cpp cacheFriendlyMatrixMulOpt.cpp
!g++ -Ofast -funroll-loops -march=native cacheFriendlyMatrixMulOpt.cpp -o cacheFriendlyMatrixMulOpt
!./cacheFriendlyMatrixMulOpt

Time taken for matrix multiplication of size 2000x2000: 1.33693 seconds


**WOW!**  Now we're really flying?!  If you have extra time, take this a step further!  
1. Can you write a program that's even more cache aware?
2. What about benchmarking your CUDA implementation?  Is it better/worse?  Why so?

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Submit to Gradescope

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)